In [3]:
pip install cooper-optim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.7/48.7 kB 3.7 MB/s eta 0:00:00


In [4]:
import numpy as np
import torch
import torch.nn as nn
import cooper
from cooper.penalty_coefficients import MultiplicativePenaltyCoefficientUpdater

In [5]:
def compute_d(arch):
    d = 0
    for i in range(len(arch) - 1):
        d += arch[i] * arch[i + 1]
    return d

In [6]:
def unpack_weights(w_flat, arch):
    matrices = []
    idx = 0
    for i in range(len(arch) - 1):
        n_in, n_out = arch[i], arch[i + 1]
        size = n_in * n_out
        W = w_flat[idx: idx + size].reshape(n_out, n_in)
        matrices.append(W)
        idx += size
    return matrices

In [7]:
def forward_pass(u, w_flat, biases, arch):
    weight_mats = unpack_weights(w_flat, arch)
    a = u
    for l, W in enumerate(weight_mats):
        z = W @ a + biases[l].unsqueeze(1)
        if l < len(weight_mats) - 1:
            a = torch.tanh(z)
        else:
            a = z
    return a

In [8]:
def binary_entropy(Q, eps=1e-6):
    Qc = Q.clamp(eps, 1 - eps) #
    return -(Qc * Qc.log() + (1 - Qc) * (1 - Qc).log()).sum()

In [9]:
class SparseNNModel(nn.Module):
    def __init__(self, arch, k, eps=1e-6):
        super().__init__()
        self.arch = arch
        self.k = k
        self.eps = eps
        self.d = compute_d(arch)

        # learnable unconstrained matrix
        self.Q_logits = nn.Parameter(torch.randn(self.d, k) * 0.01)

        # reduced coefficients
        self.x_coeff = nn.Parameter(torch.randn(k) * 0.05)

        # biases
        self.biases = nn.ParameterList()
        for i in range(len(arch) - 1):
            self.biases.append(nn.Parameter(torch.zeros(arch[i + 1])))

    def get_Q(self):
        # softmax over rows -> each column sums to 1
        return torch.softmax(self.Q_logits, dim=0)

    def get_w(self):
        Q = self.get_Q()
        return Q @ self.x_coeff

    def forward(self, u):
        w = self.get_w()
        return forward_pass(u, w, self.biases, self.arch)

In [19]:
class SparseNNCMP(cooper.ConstrainedMinimizationProblem):
    def __init__(
        self,
        model,
        c0,
        u_data,
        y_data,
        ent_tol=1e-1,
        init_penalty_entropy=1.0,
    ):
        super().__init__()
        self.model = model
        self.c0 = c0
        self.u_data = u_data
        self.y_data = y_data
        self.ent_tol = ent_tol

        self.entropy_eq = cooper.Constraint(
            constraint_type=cooper.ConstraintType.INEQUALITY,
            formulation_type=cooper.formulations.AugmentedLagrangian,
            multiplier=cooper.multipliers.DenseMultiplier(num_constraints=1),
            penalty_coefficient=cooper.penalty_coefficients.DensePenaltyCoefficient(
                init=torch.tensor([init_penalty_entropy], dtype=torch.float32)
            ),
        )

    def compute_cmp_state(self):
        m = self.u_data.shape[1]
        y_pred = self.model(self.u_data)
        loss = ((self.y_data - y_pred) ** 2).sum() / m

        Q = self.model.get_Q()
        H = binary_entropy(Q, eps=self.model.eps)
        entropy_violation = (H - self.c0).reshape(1)

        return cooper.CMPState(
            loss=loss,
            observed_constraints={
                self.entropy_eq: cooper.ConstraintState(violation=entropy_violation),
            },
        )


In [22]:
def run_homotopy(
    arch,
    u_data,
    y_data,
    k,
    c0_start,
    beta=0.97,
    outer_iters=105,
    inner_iters=300,
    ent_tol=1e-1,
    primal_lr=1e-3,
    dual_lr=1e-4,
    eps=1e-6,
    penalty_growth=1.2,
    init_penalty_entropy=1.0,
):
    torch.manual_seed(42)

    d = compute_d(arch)
    print(f"Architecture: {arch}, d={d}, k={k}")
    print(f"c0_start={c0_start:.4f}, beta={beta}, outer={outer_iters}, inner={inner_iters}")

    model = SparseNNModel(arch, k, eps=eps)
    print(f"Initial entropy H(Q) = {binary_entropy(model.get_Q(), eps).item():.4f}")

    cmp = SparseNNCMP(
        model=model,
        c0=c0_start,
        u_data=u_data,
        y_data=y_data,
        ent_tol=ent_tol,
        init_penalty_entropy=init_penalty_entropy,
    )

    primal_opt = torch.optim.Adam(model.parameters(), lr=primal_lr)
    dual_opt = torch.optim.SGD(cmp.dual_parameters(), lr=dual_lr, maximize=True)

    cooper_opt = cooper.optim.AlternatingPrimalDualOptimizer(
        cmp=cmp,
        primal_optimizers=primal_opt,
        dual_optimizers=dual_opt,
    )

    entropy_updater = MultiplicativePenaltyCoefficientUpdater(
        growth_factor=penalty_growth,
        violation_tolerance=ent_tol,
        has_restart=True,
    )

    history = []
    c0 = c0_start

    for outer in range(outer_iters):
        cmp.c0 = c0

        roll_out = None
        for _ in range(inner_iters):
            roll_out = cooper_opt.roll()

        obs = roll_out.cmp_state.observed_constraints
        entropy_updater.step({
            cmp.entropy_eq: obs[cmp.entropy_eq],
        })

        with torch.no_grad():
            Q_now = model.get_Q().detach().clone()
            H_val = binary_entropy(Q_now, eps).item()
            colsum_err = (Q_now.sum(dim=0) - 1.0).abs().max().item()
            y_pred = model(u_data)
            loss_val = ((y_data - y_pred) ** 2).sum().item() / u_data.shape[1]
            gap = torch.minimum(Q_now, 1.0 - Q_now).max().item()
            cE = cmp.entropy_eq.penalty_coefficient.value.item()

        history.append({
            "outer": outer,
            "c0": c0,
            "H": H_val,
            "loss": loss_val,
            "gap": gap,
            "colsum_err": colsum_err,
            "cE": cE,
        })

        print(f"\n=== After outer iteration {outer} ===")
        print(f"c0 = {c0:.4f}, H(Q) = {H_val:.4f}, loss = {loss_val:.4e}")
        print(f"gap = {gap:.4e}, colsum_err = {colsum_err:.4e}")
        print(f"Penalty: cE = {cE:.4f}")

        # print(f"\nSelected gradients for outer iteration {outer}:")
        # cmp.print_selected_gradients(rows=(50, 56, 111, 114), cols=(0, 1, 2, 3))

        c0 *= beta

    return model, history

In [23]:
def main():
    np.random.seed(42)

    m = 500
    u_np = np.random.uniform(-2, 2, size=(1, m))
    y_np = u_np**3 + u_np**2 + 1.0 + 0.1 * np.random.randn(1, m)

    u_data = torch.tensor(u_np, dtype=torch.float32)
    y_data = torch.tensor(y_np, dtype=torch.float32)

    arch = [1, 10, 10, 1]
    k = 20
    d = compute_d(arch)
    print(f"d = {d} (should be 120)")

    tmp_model = SparseNNModel(arch, k)
    c0_start = 0.99 * binary_entropy(tmp_model.get_Q()).item()
    del tmp_model

    print(f"Starting c0 = {c0_start:.4f}")

    model, history = run_homotopy(
        arch=arch,
        u_data=u_data,
        y_data=y_data,
        k=k,
        c0_start=c0_start,
        beta=0.97,
        outer_iters=105,
        inner_iters=300,
        primal_lr=1e-3,
        dual_lr=1e-4,
        eps=1e-6,
        ent_tol=1e-1,
        penalty_growth=1.2,
        init_penalty_entropy=1.0,
    )

    Q_final = model.get_Q().detach().cpu().numpy()
    x_final = model.x_coeff.detach().cpu().numpy()
    w_final = Q_final @ x_final

    print("\n" + "=" * 60)
    print("FINAL RESULTS")
    print("=" * 60)

    if len(history) > 0:
        print(f"Final Q binary gap: {np.max(np.minimum(Q_final, 1.0 - Q_final)):.6f}")
        print(f"Final colsum error: {np.max(np.abs(Q_final.sum(axis=0) - 1.0)):.6e}")
        print(f"Final entropy H(Q): {binary_entropy(model.get_Q()).item():.6f}")
        print(f"Final loss: {history[-1]['loss']:.6f}")

    print("\nEntries of Q > 0.9:")
    rows, cols = np.where(Q_final > 0.9)
    for r, c in zip(rows, cols):
        print(f"  Q[{r},{c}] = {Q_final[r, c]:.6f}")


if __name__ == "__main__":
    main()

d = 120 (should be 120)
Starting c0 = 114.5086
Architecture: [1, 10, 10, 1], d=120, k=20
c0_start=114.5086, beta=0.97, outer=105, inner=300
Initial entropy H(Q) = 115.6653

=== After outer iteration 0 ===
c0 = 114.5086, H(Q) = 114.6448, loss = 1.2917e+01
gap = 1.4305e-02, colsum_err = 3.5763e-07
Penalty: cE = 1.2000

=== After outer iteration 1 ===
c0 = 111.0734, H(Q) = 111.5873, loss = 1.0023e+01
gap = 1.8668e-02, colsum_err = 3.5763e-07
Penalty: cE = 1.4400

=== After outer iteration 2 ===
c0 = 107.7412, H(Q) = 108.2026, loss = 4.6347e+00
gap = 2.9387e-02, colsum_err = 2.3842e-07
Penalty: cE = 1.7280

=== After outer iteration 3 ===
c0 = 104.5089, H(Q) = 104.4212, loss = 3.3122e+00
gap = 3.8487e-02, colsum_err = 3.5763e-07
Penalty: cE = 1.0000

=== After outer iteration 4 ===
c0 = 101.3737, H(Q) = 101.1192, loss = 1.9553e+00
gap = 4.4357e-02, colsum_err = 2.3842e-07
Penalty: cE = 1.0000

=== After outer iteration 5 ===
c0 = 98.3325, H(Q) = 98.0153, loss = 1.0058e+00
gap = 5.2807e-02,